In [1]:
# New version of the LQG simulation script

import numpy as np
from src.LQG_controller import compute_lqr_gain, finite_horizon_kalman_gains
from src.simulate_lqg import simulate_lqg, simulate_robust_lqg
from src.sdp_solver import solve_sdp

# --- Parameters ---
time_horizon = 30.0  # Total time 
dt = 1.0   # Time step
T = int(time_horizon / dt)  # Number of steps 

# System dimensions
n = 2  # Number of states
m = 1  # Number of control inputs

# --- System Matrices ---
theta = np.diag([0.15, 0.1])  #	mean reversion coefficient for each state
A = np.eye(n) - theta * dt     # Dynamics matrix (n x n)
B = -dt * np.ones((n, m))      # Control input matrix (n x m)
C = np.eye(n)               # Measurement matrix (p x n) (measures first state)

# --- Initial State and Noise Covariances ---
mu = np.array([[130.0], [130.0]]) 
sigma_x0 = 2.0                     		# Initial state uncertainty
X0_hat = sigma_x0**2 * np.eye(n)   		# Initial state covariance (n x n)
sigma_w = 2.0                      		# Process noise intensity
W_hat = sigma_w**2 * dt * np.eye(n) 	# Process noise covariance (n x n)
sigma_v = 2.0                       	# Measurement noise intensity
V_hat = sigma_v**2 * np.eye(n)      	# Measurement noise covariance (p x p) 

# --- Cost Matrices ---
Q = 10 * np.eye(n)  # State cost (penalize state deviations)
R = 15 * np.eye(m)  # Control cost (penalize large inputs)

# --- Compute Nominal LQR and Kalman Gains ---
K, _ = compute_lqr_gain(A, B, Q, R)          # LQR gain (m x n)
L_list, _ = finite_horizon_kalman_gains(A, C, W_hat, V_hat, X0_hat, T)

# --- Solve SDP for Robust Covariances ---
sdp_result = solve_sdp(A, B, C, X0_hat, W_hat, V_hat, dt, T)
X0_robust = sdp_result["X0"]  # Robust initial covariance (n x n)
W0_robust = sdp_result["W0"]  # Robust process noise covariance (n x n)
V0_robust = sdp_result["V0"]  # Robust measurement noise covariance (p x p)

# --- Compute Robust Kalman Gain ---
L_rob, _ = finite_horizon_kalman_gains(A, C, W0_robust, V0_robust, X0_robust, T) 



In [3]:
print("SDP status:", sdp_result["status"])
print("\nOptimal value:", sdp_result["optimal_value"])
print("X0_robust:\n", X0_robust)
print("W0_robust:\n", W0_robust)
print("V0_robust:\n", V0_robust)


SDP status: optimal

Optimal value: 1332.9133753615824
X0_robust:
 [[ 6.7703041  -1.75652436]
 [-1.75652436  7.20041546]]
W0_robust:
 [[ 6.76040086 -1.81685894]
 [-1.81685894  7.15717057]]
V0_robust:
 [[5.60916448 2.32975464]
 [2.32975464 7.39924614]]


In [ ]:
# --- Simulate the System ---
# To compute the inital state as it is also subject to some noies (e.g., x0 = mu + noise)
x0 = np.random.multivariate_normal(mu.flatten(), X0_robust).reshape(-1, 1)

# Simulate nominal LQG
states_nominal, controls_nominal = simulate_lqg(A, B, C, K, L_list, x0, W_hat, V_hat, T)

# Simulate robust LQG
states_robust, controls_robust = simulate_robust_lqg(A, B, C, K, L_rob, x0, W0_robust, V0_robust, T)

In [ ]:
# import numpy as np
# from src.LQG_controller import compute_lqr_gain, compute_kalman_gain
# from src.simulate_lqg import simulate_lqg, simulate_robust_lqg
# from src.sdp_solver import solve_sdp

# # --- Parameters ---
# T = 30.0
# dt = 1.0
# N = int(T / dt)

# # In case we take a multidimensional system these constants have to be transformed in matrices and vectors
# n = 1 # Number of states
# m = 1 # Number of inputs
# mu = [130.0] * np.ones((n, 1)) # Mean of the state
# x0 = [120.0] * np.ones((n, 1)) # Initial state

# # System matrices (example for 2D)
# theta = np.diag([0.15, 0.1])  # Diagonal matrix for n-dimensional system
# A = np.eye(n) - theta * dt
# B = -dt * np.ones((n, m))  
# C = np.eye(p, n) 


# sigma_x0 = [2.0] # Initial state covariance
# sigma_x0 = np.array(sigma_x0).reshape((n, 1))
# sigma_w = [2.0] # Process noise
# sigma_w = np.array(sigma_w).reshape((n, 1))
# sigma_v = [2.0] # Measurement noise
# sigma_v = np.array(sigma_v).reshape((m, 1))
# # --- Cost Matrices ---
# Q = 10 * np.eye(n) # State cost
# R = 15 * np.eye(m) # Control cost

# # a = 1 - theta * dt
# # b = -dt
# # c = 1.0

# # A = np.array([[a]])
# # B = np.array([[b]])
# # C = np.array([[c]])

# # --- LQG Gain ---
# K, _ = compute_lqr_gain(A, B, Q, R)
# Qw = sigma_w**2 * dt # Process noise covariance
# Rv = sigma_v**2 # Measurement noise covariance
# L, _ = compute_kalman_gain(A, C, Qw, Rv, sigma_x0)

# # --- Robust Gains from SDP ---
# X0_opt, W0_opt, V0_opt = solve_sdp(A, B, C, sigma_x0, sigma_w, sigma_v, dt, int(T/dt))

# L_rob, _ = compute_kalman_gain(A, C, W0_opt[0][0], V0_opt[0][0], X0_opt[0][0])



In [ ]:
# # --- Simulate ---
# x, *_ = simulate_lqg(N, a, b, c, mu, K, L, sigma_x0, sigma_w, sigma_v, dt, x0)
# x_rob, *_ = simulate_robust_lqg(N, a, b, c, mu, K, L_rob, W0_opt[0][0], V0_opt[0][0], dt, x0)



In [ ]:
# --- Plot Results (optional) ---
import matplotlib.pyplot as plt
plt.plot(x, label="LQG")
plt.plot(x_rob, label="Robust LQG")
plt.legend()
plt.title("LQG vs Robust LQG")
plt.show()
